In [4]:
import os
import torch
import torchaudio
import glob
import random
import re
import hashlib
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

cuda


In [6]:
# Download the dataset
!wget -q http://download.tensorflow.org/data/speech_commands_v0.01.tar.gz

# Create a directory and extract the files
!mkdir -p /content/speech_commands
!tar -xf speech_commands_v0.01.tar.gz -C /content/speech_commands

# Remove the tar file to save local disk space
!rm speech_commands_v0.01.tar.gz

In [13]:
bg_noise_dir = '/content/speech_commands/_background_noise_'
silence_dir = '/content/speech_commands/_silence_'
os.makedirs(silence_dir, exist_ok=True)

# 1 second of audio at 16kHz
chunk_length = 16000

# Move forward by 0.1 seconds (10% of a second) to create overlapping chunks
stride = 1600

chunk_idx = 0
for filename in os.listdir(bg_noise_dir):
    if filename.endswith('.wav'):
        filepath = os.path.join(bg_noise_dir, filename)
        waveform, sample_rate = torchaudio.load(filepath)

        # Sliding window extraction
        for i in range(0, waveform.shape[1] - chunk_length, stride):
            chunk = waveform[:, i : i + chunk_length]
            torchaudio.save(os.path.join(silence_dir, f'silence_{chunk_idx}.wav'), chunk, sample_rate)
            chunk_idx += 1

print(f"Generated {chunk_idx} overlapping silence clips.")

Generated 3936 overlapping silence clips.


In [14]:
MAX_NUM_WAVS_PER_CLASS = 2**27 - 1  # ~134M

def which_set(filename, validation_percentage, testing_percentage):
    base_name = os.path.basename(filename)
    hash_name = re.sub(r'_nohash_.*$', '', base_name)

    # Python 3 fix: encode string to bytes before hashing
    hash_name_hashed = hashlib.sha1(hash_name.encode('utf-8')).hexdigest()

    percentage_hash = ((int(hash_name_hashed, 16) %
                        (MAX_NUM_WAVS_PER_CLASS + 1)) *
                       (100.0 / MAX_NUM_WAVS_PER_CLASS))

    if percentage_hash < validation_percentage:
        return 'validation'
    elif percentage_hash < (testing_percentage + validation_percentage):
        return 'testing'
    else:
        return 'training'

In [15]:
class SpeechCommands12Class(Dataset):
    def __init__(self, root_dir, wanted_words, split='training', max_unknown=None, max_silence=None):
        self.root_dir = root_dir
        self.split = split
        self.wanted_words = wanted_words
        self.classes = wanted_words + ['unknown', 'silence']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        # Adjust default sampling limits based on the split size (80% / 10% / 10%)
        if max_unknown is None:
            max_unknown = 2000 if split == 'training' else 250
        if max_silence is None:
            max_silence = 2000 if split == 'training' else 250

        # Load the official validation and testing lists
        with open(os.path.join(root_dir, 'validation_list.txt'), 'r') as f:
            val_list = set(f.read().splitlines())
        with open(os.path.join(root_dir, 'testing_list.txt'), 'r') as f:
            test_list = set(f.read().splitlines())

        target_data = []
        unknown_data = []
        silence_data = []

        all_wavs = glob.glob(os.path.join(root_dir, '*/*.wav'))

        for filepath in all_wavs:
            folder = os.path.basename(os.path.dirname(filepath))
            filename = os.path.basename(filepath)

            # Skip the original raw noise folder
            if folder == '_background_noise_':
                continue

            # Determine which split this file belongs to
            relative_path = f"{folder}/{filename}"

            if relative_path in val_list:
                file_split = 'validation'
            elif relative_path in test_list:
                file_split = 'testing'
            elif folder == '_silence_':
                # Use the hash function for our custom silence files
                file_split = which_set(filename, validation_percentage=10.0, testing_percentage=10.0)
            else:
                file_split = 'training'

            # Only collect files that match the requested split
            if file_split != self.split:
                continue

            # Assign labels
            if folder in wanted_words:
                target_data.append((filepath, self.class_to_idx[folder]))
            elif folder == '_silence_':
                silence_data.append((filepath, self.class_to_idx['silence']))
            else:
                unknown_data.append((filepath, self.class_to_idx['unknown']))

        # Downsample unknown and silence classes
        random.seed(2104)
        if len(unknown_data) > max_unknown:
            unknown_data = random.sample(unknown_data, max_unknown)
        if len(silence_data) > max_silence:
            silence_data = random.sample(silence_data, max_silence)

        all_data = target_data + unknown_data + silence_data
        random.shuffle(all_data)

        self.filepaths = [item[0] for item in all_data]
        self.labels = [item[1] for item in all_data]

        # Audio transforms
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=1024, hop_length=512, n_mels=64
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        label = self.labels[idx]

        waveform, sample_rate = torchaudio.load(filepath)

        if waveform.shape[1] < 16000:
            waveform = F.pad(waveform, (0, 16000 - waveform.shape[1]))
        elif waveform.shape[1] > 16000:
            waveform = waveform[:, :16000]

        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)

        return mel_spec_db, torch.tensor(label, dtype=torch.long)

In [16]:
WANTED_WORDS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
DATA_DIR = '/content/speech_commands'

# Instantiate the three precise splits
train_dataset = SpeechCommands12Class(DATA_DIR, WANTED_WORDS, split='training')
val_dataset   = SpeechCommands12Class(DATA_DIR, WANTED_WORDS, split='validation')
test_dataset  = SpeechCommands12Class(DATA_DIR, WANTED_WORDS, split='testing')

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

Training samples: 22538
Validation samples: 3077
Testing samples: 3067


In [19]:
from collections import Counter

# Count the occurrences of each label index in the dataset
label_counts = Counter(test_dataset.labels)

print(f"Total samples in training set: {len(test_dataset)}")
print("-" * 35)

# Print them out neatly, mapped back to their human-readable names
for idx, count in sorted(label_counts.items()):
    class_name = train_dataset.classes[idx]

    # We use formatting to align the text nicely
    print(f"{class_name:>10}: {count} samples")

Total samples in training set: 3067
-----------------------------------
       yes: 256 samples
        no: 252 samples
        up: 272 samples
      down: 253 samples
      left: 267 samples
     right: 259 samples
        on: 246 samples
       off: 262 samples
      stop: 249 samples
        go: 251 samples
   unknown: 250 samples
   silence: 250 samples
